# Prompt Injection Detector — DistilBERT (Colab)
Upload the `prompt_injection.csv` created by `prepare_data.py`. In Colab, choose Runtime → Change runtime type → GPU (availability varies). Labels: 0 benign, 1 injection. Scores are a prototype evaluation, not a guarantee against novel attacks.

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn pandas

In [ ]:
from google.colab import files
uploaded=files.upload()
import pandas as pd, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
csv_path=next(iter(uploaded))
df=pd.read_csv(csv_path)[['text','label']].dropna()
df.text=df.text.astype(str).str.strip()
df.label=pd.to_numeric(df.label,errors='coerce')
df=df.dropna().query('label in [0,1]')
df.label=df.label.astype(int)
df=df[df.text.str.len().between(4,4000)].drop_duplicates('text').reset_index(drop=True)
print('rows',len(df),'classes',df.label.value_counts().to_dict())

## Split and fine-tune
A random split can overstate performance when similar templates occur across partitions. For a stronger report, group by source or attack template before splitting.

In [ ]:
seed=42
train_df, rest=train_test_split(df,test_size=.2,stratify=df.label,random_state=seed)
val_df,test_df=train_test_split(rest,test_size=.5,stratify=rest.label,random_state=seed)
base='distilbert/distilbert-base-uncased'
tok=AutoTokenizer.from_pretrained(base)
def encode(frame):
    ds=Dataset.from_pandas(frame.reset_index(drop=True)).rename_column('label','labels')
    return ds.map(lambda x:tok(x['text'],truncation=True,max_length=256),batched=True,remove_columns=['text'])
tr,va,te=map(encode,[train_df,val_df,test_df])
model=AutoModelForSequenceClassification.from_pretrained(base,num_labels=2,id2label={0:'BENIGN',1:'INJECTION'},label2id={'BENIGN':0,'INJECTION':1})
def metrics(p):
    logits,y=p; scores=torch.softmax(torch.tensor(logits),dim=1).numpy()[:,1]; pred=(scores>=.5).astype(int)
    return {'accuracy':accuracy_score(y,pred),'precision_injection':precision_score(y,pred,zero_division=0),'recall_injection':recall_score(y,pred,zero_division=0),'f1_injection':f1_score(y,pred,zero_division=0),'roc_auc':roc_auc_score(y,scores)}
args=TrainingArguments(output_dir='/content/pi-checkpoints',learning_rate=2e-5,per_device_train_batch_size=16,per_device_eval_batch_size=32,num_train_epochs=3,weight_decay=.01,evaluation_strategy='epoch',save_strategy='epoch',load_best_model_at_end=True,metric_for_best_model='f1_injection',fp16=torch.cuda.is_available(),report_to='none',seed=seed)
trainer=Trainer(model=model,args=args,train_dataset=tr,eval_dataset=va,tokenizer=tok,data_collator=DataCollatorWithPadding(tok),compute_metrics=metrics)
trainer.train()

## Test results and security improvement
Attack block rate is injection recall. Benign false-block rate measures genuine requests rejected. An unguarded pass-through baseline blocks 0% of attacks; report the measured percentage point increase and the false-block cost. Do not tune the threshold on the test set.

In [ ]:
out=trainer.predict(te)
scores=torch.softmax(torch.tensor(out.predictions),dim=1).numpy()[:,1]
y=test_df.label.to_numpy(); threshold=.5; pred=(scores>=threshold).astype(int)
tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
attack_block=tp/(tp+fn) if tp+fn else 0
false_block=fp/(fp+tn) if fp+tn else 0
print(f'test accuracy={accuracy_score(y,pred):.4f}')
print(f'injection precision={precision_score(y,pred,zero_division=0):.4f}')
print(f'injection recall / attack block rate={attack_block:.4f}')
print(f'injection F1={f1_score(y,pred,zero_division=0):.4f}')
print(f'benign false-block rate={false_block:.4f}')
print('unguarded baseline attack block rate=0.0000')
print(f'absolute improvement={attack_block*100:.2f} percentage points; blocked {tp}/{tp+fn} attacks; false-blocked {fp}/{fp+tn} benign prompts')

## Save model ZIP

In [ ]:
import shutil
model_dir='/content/prompt-injection-distilbert'
trainer.save_model(model_dir); tok.save_pretrained(model_dir)
shutil.make_archive('prompt-injection-distilbert','zip',model_dir)
files.download('/content/prompt-injection-distilbert.zip')